# Auxotrophies and minimal medium

In [1]:
import reframed
import pandas as pd
import numpy as np
from os import listdir
import warnings
import seaborn as sns
from collections import OrderedDict
import json

In [2]:
import sys

sys.path.append("../functions/")
from minimal_medium import make_minimal_media_df

In [3]:
%load_ext autoreload
%autoreload 2

**Set gurobi as default solver**

In [4]:
reframed.set_default_solver("gurobi")

**Load other data (E. coli)**

In [5]:
relative_abundance_ecoli = pd.read_csv("../data/Guillen2021/relative_abundance.csv",index_col=0)
relative_abundance_ecoli.index = relative_abundance_ecoli.index.map(lambda x:x if x!="yhhK" else "panZ")

gene_id2name = pd.read_csv("../data/Guillen2021/generated/gene_id2name.tsv",sep="\t")

**Load models**

*E. coli mutants  (top 7) and (all 38)*

In [6]:
RA_ecoli_mean = relative_abundance_ecoli.T.mean().sort_values(ascending=False)
top7 = RA_ecoli_mean[RA_ecoli_mean>0.01].index
top7_string = [f"mutant_{mutant}.xml" for mutant in top7]

In [ ]:
models_ecoli = {}
models_ecoli_top7 = {}
with warnings.catch_warnings():
    warnings.filterwarnings('ignore')
    for path in listdir("../models/Guillen2021/strains/"):
        if not path.endswith(".xml"): continue
        model= reframed.load_cbmodel("../models/Guillen2021/strains/"+path,
                                     exchange_detection="^R_EX_(?!nad_c$|thf_c$).*")
        
        models_ecoli[path.split("_")[1][:-4]] = model
        
        if path not in top7_string: continue
        models_ecoli_top7[path.split("_")[1][:-4]] = model.copy()

*Gut microbiome members*

In [ ]:
models_gut = {}
with warnings.catch_warnings():
    warnings.filterwarnings('ignore')
    for path in listdir("../models/Venturelli2018/species_working_models/"):
        models_gut[path[:-4]] = reframed.load_cbmodel("../models/Venturelli2018/species_used_noO2/"+path,
                                                      exchange_detection="R_EX_(?!biomass__40__e__41__)")


**Load media**

*For E. coli mutants (LB medium to see auxotrophies)*

In [ ]:
media_db = pd.read_csv("https://raw.githubusercontent.com/cdanielmachado/carveme/refs/heads/master/carveme/data/input/media_db.tsv",sep="\t")
compounds = list(media_db[media_db.medium=="LB"].compound.values)
compounds.append("btn") # biot members are auxotrophic to biotin which was not in LB-medium. 
compounds.append("mmet") # metE and metH double mutant has been shown to be dependent on mmet (S-Methylmethionine). metH is missing in the model!
compounds.append("26dap__M") # meso-DAP. Essential part of cell wall in gram-negative bacteria...
env_ecoli_lb = reframed.Environment.from_compounds(compounds)

*For gut microbiome members*

In [ ]:
media_db_gut = pd.read_csv("../data/Venturelli2018/generated/media_db.tsv",sep="\t")
reactions = media_db_gut[media_db_gut.medium=="LB_extend"].compound.values
reactions = [rxn.replace("91","40").replace("93","41") for rxn in reactions]

env_gut = reframed.Environment.from_reactions(reactions)

**Translation dicts**

In [ ]:
mets_id_ag = []
mets_id_bigg = []
ex_rxns_ag = []
for model in models_gut.values():
    for rxn in model.get_exchange_reactions():
        if "R_EX_" not in rxn:
            continue
        met_id_ag = model.reactions[rxn].get_substrates()[0]
        mets_id_ag.append(met_id_ag)
        met_id_bigg = model.metabolites[met_id_ag].metadata["XMLAnnotation"].split("bigg.metabolite/")[1].split("\"")[0]
        mets_id_bigg.append(met_id_bigg)
        ex_rxns_ag.append(rxn)
    
met2ex = dict(zip(mets_id_bigg,ex_rxns_ag))

### Find minimal medium

In [ ]:
minimal_media_df_ecoli = make_minimal_media_df(models_ecoli,env_ecoli_lb)

In [ ]:
minimal_media_df_gut = make_minimal_media_df(models_gut,env_gut)

In [ ]:
minimal_media_df_gut.to_csv("../output/Venturelli2018/minimal_media.tsv",sep="\t")

minimal_media_df_ecoli.to_csv("../output/Guillen2021/minimal_media.tsv",sep="\t")

### Plot solutions

**Remove results for less interesting compounds**: *compounds that few members have auxotrophy for/compounds all members have auxotrophy for, and minerals etc (from M9 medium)*

In [ ]:
m9_mets = media_db[media_db.medium=="M9"].compound.values
m9_rxns = ["R_EX_"+met+"_e" for met in m9_mets]
m9_rxns_gut = ["R_EX_"+met+"__40__e__41__" for met in m9_mets]

In [ ]:
minimal_media_df_ecoli_plot = minimal_media_df_ecoli.loc[(minimal_media_df_ecoli.sum(axis=1)<750*minimal_media_df_ecoli.shape[1]) & (minimal_media_df_ecoli.sum(axis=1)>=900)]
minimal_media_df_ecoli_plot = minimal_media_df_ecoli_plot[~minimal_media_df_ecoli_plot.index.isin(m9_rxns)]

minimal_media_df_gut_plot = minimal_media_df_gut.loc[(minimal_media_df_gut.sum(axis=1)<750*minimal_media_df_gut.shape[1]) & (minimal_media_df_gut.sum(axis=1)>=900)]
minimal_media_df_gut_plot = minimal_media_df_gut_plot[~minimal_media_df_gut_plot.index.isin(m9_rxns_gut)]


**Define color palette**

In [ ]:
bigg_met2chebi_class = pd.read_csv("../../../GEMs_from_MAGs/output/met_chebi_class.tsv",sep="\t",index_col=0)

met2chebi = bigg_met2chebi_class["self defined super class"].to_dict()
met2chebi = {key[2:-2]:value for key,value in met2chebi.items()}

agoraEx2chebi = {met2ex[key]: value for key, value in met2chebi.items() if key in met2ex.keys()}

for value in minimal_media_df_gut.index:
    if value not in agoraEx2chebi.keys():
        agoraEx2chebi[value]="undefined"

biggEx2chebi = {"R_EX_"+met+"_e":super_class for met,super_class in met2chebi.items() if "R_EX_"+met+"_e" in minimal_media_df_ecoli_plot.index}
for value in minimal_media_df_ecoli_plot.index:
    if value not in biggEx2chebi.keys():
        biggEx2chebi[value]="undefined"


pallete = sns.color_palette("tab20b",20)
pallete.reverse()
pallete2 = sns.color_palette("tab20c",20)
pallete2.reverse()
pallete.extend([pallete2[-3],pallete2[-1]])


super_classes = list(set(agoraEx2chebi.values()))+list(set(biggEx2chebi.values()))
pallete_dict = OrderedDict(zip(super_classes,pallete))
pallete_series_agora = pd.Series({key:pallete_dict[value] for key,value in agoraEx2chebi.items()} )
pallete_series_bigg = pd.Series({key:pallete_dict[value] for key,value in biggEx2chebi.items()} )


#### Members of E.coli mutants

In [ ]:
from matplotlib.pyplot import gcf

g = sns.clustermap(minimal_media_df_ecoli_plot,
                   cmap="binary",
                   method="ward",
                   vmin=0,
                   fmt="d",
                   dendrogram_ratio=(0.2, 0.1),
                   row_colors=pallete_series_bigg,
                   xticklabels=1)

relevant_classes = list(set([biggEx2chebi[key] for key in minimal_media_df_ecoli_plot.index]))

for label in relevant_classes:
    g.ax_col_dendrogram.bar(0, 0, color=pallete_dict[label], label=label, linewidth=0)
l1 = g.ax_col_dendrogram.legend(title='Phyla',ncol=1, bbox_to_anchor=(1.0, 0.85), loc='upper left',bbox_transform=gcf().transFigure)

**Find experimental results for top 7 members**

In [ ]:
RA_ecoli_mean = relative_abundance_ecoli.T.mean().sort_values(ascending=False)
RA_ecoli_mean[RA_ecoli_mean>0.01]

In [ ]:
relevant_ecoli_mutants = RA_ecoli_mean[RA_ecoli_mean>0.01].index
minimal_media_df_ecoli_plot2 = minimal_media_df_ecoli_plot.loc[:,minimal_media_df_ecoli_plot.columns.isin(relevant_ecoli_mutants)]

g = sns.clustermap(minimal_media_df_ecoli_plot2,
                   cmap="binary",
                   method="ward",
                   vmin=0,
                   fmt="d",
                   dendrogram_ratio=(0.2, 0.1),
                   row_colors=pallete_series_bigg,
                   xticklabels=1)

relevant_classes = list(set([biggEx2chebi[key] for key in minimal_media_df_ecoli_plot2.index]))

for label in relevant_classes:
    g.ax_col_dendrogram.bar(0, 0, color=pallete_dict[label], label=label, linewidth=0)
l1 = g.ax_col_dendrogram.legend(title='Phyla',ncol=1, bbox_to_anchor=(1.0, 0.85), loc='upper left',bbox_transform=gcf().transFigure)

<span style="color:red"> NB! bio(*) strains are auxotrophic to biotin (btn) - confirmed by removing from media.  I don't understand why it doesn't show...

In [ ]:
bio_members = ["bioH","bioF","bioC","panZ"] #panZ for control


In [ ]:
env_ecoli_no_btn = env_ecoli_lb.copy()
env_ecoli_no_btn.pop("R_EX_btn_e")


In [ ]:
for mutant in bio_members:
    model = models_ecoli[mutant].copy()
    env_ecoli_no_btn.apply(model,inplace=True,exclusive=True,warning=False)
    
    sol = reframed.FBA(model)
    print(f"{mutant}: {sol.fobj}")

#### Members of gut microbiome

In [ ]:
g = sns.clustermap(minimal_media_df_gut_plot,
                   figsize=(10,12),
                   cmap="binary",
                   method="ward",
                   vmin=0,
                   fmt="d",
                   dendrogram_ratio=(0.2, 0.1),
                   row_colors=pallete_series_agora,
                   xticklabels=1)

relevant_classes = list(set([agoraEx2chebi[key] for key in minimal_media_df_gut_plot.index]))

for label in relevant_classes:
    g.ax_col_dendrogram.bar(0, 0, color=pallete_dict[label], label=label, linewidth=0)
l1 = g.ax_col_dendrogram.legend(title='Phyla',ncol=1, bbox_to_anchor=(1.0, 0.85), loc='upper left',bbox_transform=gcf().transFigure)

## Check community growth

**Load media**

*For E. coli mutants (M9 medium to see if they can grow together)*

In [ ]:
media_db = pd.read_csv("https://raw.githubusercontent.com/cdanielmachado/carveme/refs/heads/master/carveme/data/input/media_db.tsv",sep="\t")
compounds = list(media_db[media_db.medium=="M9"].compound.values)
env_ecoli_m9 = reframed.Environment.from_compounds(compounds,max_uptake=50)

*For gut microbiome members*

In [ ]:
media_db_gut = pd.read_csv("../data/Venturelli2018/generated/media_db.tsv",sep="\t")
reactions = media_db_gut[media_db_gut.medium=="LB_extend"].compound.values
env_gut = reframed.Environment.from_reactions(reactions,max_uptake=10)

### Check community growth

**Make community model**

*For E. coli mutants*

In [ ]:
comm_ecoli = reframed.Community(models=models_ecoli_top7.values(),community_id="ecoli_mutants")

*For gut microbiome members*

In [ ]:
comm_gut = reframed.Community(models=models_gut.values(),community_id="gut_microbiome")

#### Ecoli mutants community growth

In [ ]:
env_ecoli_m9.apply(comm_ecoli.merged_model, inplace=True, exclusive=True, warning=False)

biomass_rxns = comm_ecoli.merged_model.get_metabolite_producers("community_biomass")
growth = np.ones(len(biomass_rxns))*0.1
growth_upper = np.ones(len(biomass_rxns))*1000
constraints_growth_ecoli = dict(zip(biomass_rxns,list(zip(growth,growth_upper))))

In [ ]:
sol = reframed.FBA(comm_ecoli.merged_model,
                   constraints=constraints_growth_ecoli
                  )
sol.values["community_growth"]

In [ ]:
for rxn in biomass_rxns:
    print(f"{rxn}: {sol.values[rxn]}")

#### Gut microbiome community growth

In [ ]:
env_gut.apply(comm_gut.merged_model, inplace=True, exclusive=True, warning=False)

biomass_rxns = comm_gut.merged_model.get_metabolite_producers("community_biomass")
growth = np.ones(len(biomass_rxns))*0.1
growth_upper = np.ones(len(biomass_rxns))*1000
constraints_growth_gut = dict(zip(biomass_rxns,list(zip(growth,growth_upper))))

In [ ]:
sol = reframed.FBA(comm_gut.merged_model,constraints=constraints_growth_gut)

sol.values["community_growth"]

In [ ]:
for rxn in biomass_rxns:
    print(f"{rxn}: {sol.values[rxn]}")